# 06 — ETL: River Corridors

Prepares Vermont ANR River Corridor polygon data for the Floodlines dashboard.

**Output:**
- `../docs/static/resources/river_corridors_tier1.geojson` — aggressively simplified, always-on state-level context layer
- `../docs/static/resources/river_corridors_tier2.geojson` — moderately simplified, loaded lazily on zoom-in, filtered client-side to viewport bounds

**Source:** Vermont ANR River Corridors (WaterHydro_RiverCorridors_poly.shp)


In [1]:
# import dependencies
import geopandas as gpd
import json
import os

In [2]:
# load river corridor shapefile into GeoDataFrame
rc_fp = "../data/raw/VT_river_corridors/WaterHydro_RiverCorridors_poly.shp"
gdf_rc = gpd.read_file(rc_fp)

print(f"Shape:   {gdf_rc.shape}")
print(f"CRS:     {gdf_rc.crs}")
print(f"Columns: {list(gdf_rc.columns)}")
gdf_rc.head(3)

Shape:   (11093, 17)
CRS:     PROJCS["NAD83 / Vermont",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",42.5],PARAMETER["central_meridian",-72.5],PARAMETER["scale_factor",0.999964285714286],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns: ['OBJECTID', 'SGAT_ID', 'MEAN_DA_sq', 'MEAN_Bkful', 'FIRST_GNIS', 'DA_sqmi_f', 'Bkful_ft', 'ER_Power', 'DEP_Power', 'GNIS_NAME', 'P1_ChMult', 'MN_DMS_ChW', 'MNP1ChMult', 'MNP2ChMult', 'F1stER_Pow', 'F1stDEP_Po', 'geometry']


,OBJECTID,SGAT_ID,MEAN_DA_sq,MEAN_Bkful,FIRST_GNIS,DA_sqmi_f,Bkful_ft,ER_Power,DEP_Power,GNIS_NAME,P1_ChMult,MN_DMS_ChW,MNP1ChMult,MNP2ChMult,F1stER_Pow,F1stDEP_Po,geometry
0,1,103_M03A,17.5261,46.183666,Bloody Brook,0.0,0.0,None,None,None,0.0,46.316299,3.0,3.0,LOW,HIGH,"POLYGON ((515490 133920, 515490 133846.491, 51..."
1,2,103_M03B,17.2112,45.817101,Bloody Brook,0.0,0.0,None,None,None,0.0,46.316299,2.5,3.0,MODERATE,MODERATE,"POLYGON ((514850 135170, 514850 135160, 514860..."
2,3,103_M04-,15.4304,43.665001,Bloody Brook,0.0,0.0,None,None,None,0.0,44.041401,3.0,3.0,MODERATE,MODERATE,"POLYGON ((515125.901 135241.212, 515111.948 13..."


In [3]:
# reproject to WGS84 (EPSG:4326) for GeoJSON export and JS mapping
if gdf_rc.crs.to_epsg() != 4326:
    gdf_rc = gdf_rc.to_crs(epsg=4326)

print(f"CRS after reprojection: {gdf_rc.crs}")

# check raw file size for reference
raw_size_mb = os.path.getsize(rc_fp) / (1024**2)
print(f"Raw shapefile size: {raw_size_mb:.1f} MB")

CRS after reprojection: EPSG:4326
Raw shapefile size: 31.6 MB


## Select and rename columns

Keep only the fields useful for a dashboard tooltip. Everything else inflates file size.


In [4]:
# print all columns to identify what to keep
print(gdf_rc.columns.tolist())

['OBJECTID', 'SGAT_ID', 'MEAN_DA_sq', 'MEAN_Bkful', 'FIRST_GNIS', 'DA_sqmi_f', 'Bkful_ft', 'ER_Power', 'DEP_Power', 'GNIS_NAME', 'P1_ChMult', 'MN_DMS_ChW', 'MNP1ChMult', 'MNP2ChMult', 'F1stER_Pow', 'F1stDEP_Po', 'geometry']


In [5]:
# keep only columns useful for dashboard tooltips + geometry
keep_cols = [
    "SGAT_ID",  # segment ID
    "GNIS_NAME",  # stream name
    "DA_sqmi_f",  # drainage area (sq mi)
    "Bkful_ft",  # bankfull width (ft)
    "ER_Power",  # erosion power/risk
    "DEP_Power",  # deposition power/risk
    "geometry",
]
keep_cols = [c for c in keep_cols if c in gdf_rc.columns]
gdf_rc = gdf_rc[keep_cols]

print(f"Retained columns: {list(gdf_rc.columns)}")
print(f"Feature count:    {len(gdf_rc)}")

Retained columns: ['SGAT_ID', 'GNIS_NAME', 'DA_sqmi_f', 'Bkful_ft', 'ER_Power', 'DEP_Power', 'geometry']
Feature count:    11093


## Simplify geometries

- **Tier 1** (`0.003°` ≈ 300m): state-level context, always loaded at startup; keep small
- **Tier 2** (`0.00005°` ≈ 5m): visually indistinguishable from raw; lazy-loaded on zoom-in, viewport-filtered on the client so render cost stays constant regardless of file size

Note: no-simplification GeoJSON is ~89 MB (shapefile binary encodes much more compactly than JSON text).


In [6]:
# simplify geometries to create tiered datasets for faster rendering at different zoom levels
TIER1_TOL = 0.003  # ~300m — state-level context, always loaded at startup
TIER2_TOL = (
    0.00005  # ~5m   — visually indistinguishable from raw; lazy + viewport-filtered
)

# create simplified GeoDataFrames for each tier, dropping any invalid geometries that may result from simplification
gdf_tier1 = gdf_rc.copy()
gdf_tier1["geometry"] = gdf_tier1.geometry.simplify(TIER1_TOL, preserve_topology=True)
gdf_tier1 = gdf_tier1[gdf_tier1.geometry.notna() & ~gdf_tier1.geometry.is_empty]

gdf_tier2 = gdf_rc.copy()
gdf_tier2["geometry"] = gdf_tier2.geometry.simplify(TIER2_TOL, preserve_topology=True)
gdf_tier2 = gdf_tier2[gdf_tier2.geometry.notna() & ~gdf_tier2.geometry.is_empty]

print(f"Tier 1: {len(gdf_tier1)} features  (tol={TIER1_TOL}°)")
print(f"Tier 2: {len(gdf_tier2)} features  (tol={TIER2_TOL}°)")

Tier 1: 11093 features  (tol=0.003°)
Tier 2: 11093 features  (tol=5e-05°)


## Export


In [7]:
# export simplified GeoDataFrames to GeoJSON files for use in the dashboard
OUT_DIR = "../docs/static/resources"
os.makedirs(OUT_DIR, exist_ok=True)

tier1_path = os.path.join(OUT_DIR, "river_corridors_tier1.geojson")
tier2_path = os.path.join(OUT_DIR, "river_corridors_tier2.geojson")

gdf_tier1.to_file(tier1_path, driver="GeoJSON")
gdf_tier2.to_file(tier2_path, driver="GeoJSON")

# report output file sizes
for path in [tier1_path, tier2_path]:
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"{os.path.basename(path)}: {size_mb:.2f} MB")

river_corridors_tier1.geojson: 4.93 MB
river_corridors_tier2.geojson: 18.55 MB
